# 📊 Análisis de ventas y marketing con Python
### Limpieza de datos, análisis exploratorio, estadística descriptiva y dashboard interactivo

**Autor:** Alejandro Calello · [LinkedIn](https://www.linkedin.com/in/alejandrocalello) · [GitHub](https://github.com/AlejandroCalello)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlejandroCalello/analisis-ventas-marketing-python/blob/main/analisis_ventas_marketing.ipynb)

**Objetivo:** entender qué productos y categorías generan los ingresos, cómo se distribuyen, si el precio o la inversión en marketing se relacionan con las ventas, y presentar los resultados en un dashboard interactivo.

**Datos** (3 archivos CSV en la carpeta `data/`):

| Archivo | Contenido |
|---|---|
| `ventas.csv` | Ventas individuales: producto, categoría, precio, cantidad y fecha |
| `marketing.csv` | Campañas por producto: canal, costo, fechas de inicio y fin |
| `clientes.csv` | Clientes: edad, ciudad e ingresos |

**Herramientas:** Python · pandas · NumPy · Matplotlib · Seaborn · SciPy · Plotly

**Contenido**
1. Carga y exploración
2. Calidad de los datos
3. Limpieza y transformación
4. Agregación e integración (ventas + marketing)
5. Concentración de ingresos (Pareto)
6. Estadística descriptiva y outliers
7. Correlaciones
8. Evolución en el tiempo
9. Marketing vs. ingresos
10. Dashboard interactivo
11. Conclusiones

## 1. Carga y exploración inicial

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format

La función `leer_csv` busca cada archivo en este orden: carpeta local `data/`, el repositorio de GitHub (para que el notebook corra directo desde Colab) y, como último recurso, Google Drive.

In [ ]:
CARPETA_LOCAL = Path("data")
URL_BASE = "https://raw.githubusercontent.com/AlejandroCalello/analisis-ventas-marketing-python/main/data/"


def leer_csv(nombre):
    ruta = CARPETA_LOCAL / nombre
    if ruta.exists():                      # 1) carpeta local
        return pd.read_csv(ruta)
    try:                                   # 2) repositorio en GitHub
        return pd.read_csv(URL_BASE + nombre)
    except Exception:
        pass
    try:                                   # 3) Google Drive (solo en Colab)
        from google.colab import drive
        drive.mount("/content/drive")
        return pd.read_csv(f"/content/drive/MyDrive/datasets/{nombre}")
    except ImportError:
        raise FileNotFoundError(
            f"No encuentro {nombre}. Ponelo en la carpeta data/ junto al notebook."
        )


ventas = leer_csv("ventas.csv")
marketing = leer_csv("marketing.csv")
clientes = leer_csv("clientes.csv")

print(f"ventas: {ventas.shape} | marketing: {marketing.shape} | clientes: {clientes.shape}")

In [ ]:
def explorar(df, nombre):
    print(f"===== {nombre.upper()} =====")
    print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
    df.info()
    display(df.head(3))
    display(df.describe())


explorar(ventas, "ventas")

In [ ]:
explorar(marketing, "marketing")

In [ ]:
explorar(clientes, "clientes")

**Observaciones iniciales**
- `precio` viene como texto (con símbolo `$` y comas), por lo que hay que convertirlo a número.
- Las fechas vienen como texto y hay que convertirlas a `datetime`.
- `cantidad` viene como decimal aunque son unidades enteras.
- Los nombres de producto de `ventas` y de `marketing` no tienen el mismo formato (mayúsculas y espacios), lo que afecta al unir las tablas.
- `ventas` no tiene `id_cliente`, así que **no se puede vincular una venta con un cliente**. Por eso `clientes` solo se explora y no entra en el análisis de ventas.

## 2. Calidad de los datos

In [ ]:
def calidad(df, nombre):
    print(f"--- {nombre} ---")
    nulos = df.isna().sum()
    print("Nulos por columna:")
    print(nulos[nulos > 0] if nulos.sum() else "  (sin nulos)")
    print(f"Filas duplicadas: {df.duplicated().sum()}\n")


calidad(ventas, "ventas")
calidad(marketing, "marketing")
calidad(clientes, "clientes")

## 3. Limpieza y transformación

Trabajo sobre copias para no tocar los datos originales. Pasos:
1. Eliminar duplicados exactos (antes de imputar, para que no distorsionen los promedios).
2. Normalizar textos: mayúsculas, espacios y **la misma regla en ventas y marketing**.
3. Convertir fechas, precios y cantidades a su tipo correcto.
4. Tratar nulos: el precio se imputa con el promedio del mismo producto; las filas sin cantidad se descartan, porque no se puede inferir cuántas unidades se vendieron.
5. Crear `total = precio × cantidad`.

In [ ]:
ventas_clean = ventas.copy()
clientes_clean = clientes.copy()
marketing_clean = marketing.copy()

# 1) Duplicados exactos
n_duplicados = int(ventas_clean.duplicated().sum())
ventas_clean = ventas_clean.drop_duplicates()
clientes_clean = clientes_clean.drop_duplicates()
marketing_clean = marketing_clean.drop_duplicates()
print(f"Duplicados eliminados en ventas: {n_duplicados}")

In [ ]:
# 2) Normalización de textos (misma función para ventas y marketing)
def normalizar_texto(serie):
    return (
        serie.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)   # varios espacios -> uno
        .str.title()
    )


ventas_clean["producto"] = normalizar_texto(ventas_clean["producto"])
ventas_clean["categoria"] = normalizar_texto(ventas_clean["categoria"])
marketing_clean["producto"] = normalizar_texto(marketing_clean["producto"])
marketing_clean["canal"] = normalizar_texto(marketing_clean["canal"])

# Cada producto debería pertenecer a una sola categoría
assert ventas_clean.groupby("producto")["categoria"].nunique().max() == 1

In [ ]:
# 3) Tipos de datos
def a_fecha(serie):
    # Formato ISO (2024-03-15) o día/mes/año (15/03/2024). Con ISO, dayfirst=True da fechas mal leídas.
    es_iso = serie.astype(str).str.match(r"^\d{4}-\d{2}-\d{2}").all()
    if es_iso:
        return pd.to_datetime(serie, errors="coerce", format="ISO8601")
    return pd.to_datetime(serie, errors="coerce", dayfirst=True)


ventas_clean["fecha_venta"] = a_fecha(ventas_clean["fecha_venta"])
marketing_clean["fecha_inicio"] = a_fecha(marketing_clean["fecha_inicio"])
marketing_clean["fecha_fin"] = a_fecha(marketing_clean["fecha_fin"])
print("Fechas que no se pudieron convertir:",
      int(ventas_clean["fecha_venta"].isna().sum() + marketing_clean[["fecha_inicio", "fecha_fin"]].isna().sum().sum()))

# Precio: quitar $, comas y espacios, y pasar a número
ventas_clean["precio"] = pd.to_numeric(
    ventas_clean["precio"].astype(str).str.replace(r"[$,\s]", "", regex=True),
    errors="coerce",
)
ventas_clean["cantidad"] = pd.to_numeric(ventas_clean["cantidad"], errors="coerce")

In [ ]:
# 4) Nulos
n_precio_nulo = int(ventas_clean["precio"].isna().sum())
n_cantidad_nula = int(ventas_clean["cantidad"].isna().sum())

# Precio faltante -> promedio de precios del mismo producto
ventas_clean["precio"] = ventas_clean["precio"].fillna(
    ventas_clean.groupby("producto")["precio"].transform("mean")
)

# Cantidad faltante -> se descarta la fila (no sabemos cuántas unidades fueron)
ventas_clean = ventas_clean.dropna(subset=["cantidad"]).copy()
ventas_clean["cantidad"] = ventas_clean["cantidad"].astype(int)

print(f"Precios imputados: {n_precio_nulo} | Filas descartadas por cantidad nula: {n_cantidad_nula}")
assert ventas_clean.isna().sum().sum() == 0, "Quedaron nulos en ventas_clean"

# 5) Ingreso por venta
ventas_clean["total"] = ventas_clean["precio"] * ventas_clean["cantidad"]
ventas_clean.head()

In [ ]:
# Chequeo final
for nombre, df in [("ventas", ventas_clean), ("marketing", marketing_clean), ("clientes", clientes_clean)]:
    print(f"{nombre}: {df.shape[0]:,} filas | nulos: {int(df.isna().sum().sum())} | duplicados: {int(df.duplicated().sum())}")
ventas_clean.dtypes

In [ ]:
# Distribución del ingreso por venta individual
plt.figure(figsize=(8, 5))
plt.hist(ventas_clean["total"], bins=10, edgecolor="black", color="#4C78A8")
plt.title("Distribución del ingreso por venta")
plt.xlabel("Ingreso por venta ($)")
plt.ylabel("Cantidad de ventas")
plt.tight_layout()
plt.show()

## 4. Agregación e integración (ventas + marketing)

In [ ]:
# Ventas por producto
ventas_por_producto = (
    ventas_clean
    .groupby("producto", as_index=False)
    .agg(
        total=("total", "sum"),
        cantidad=("cantidad", "sum"),
        precio=("precio", "mean"),
        categoria=("categoria", "first"),
        registros=("id_venta", "count"),
    )
    .sort_values("total", ascending=False)
    .reset_index(drop=True)
)

total_general = ventas_por_producto["total"].sum()
print(f"Ingresos totales: ${total_general:,.2f} | Productos: {len(ventas_por_producto)}")
ventas_por_producto.head(10)

In [ ]:
# Ventas por categoría
ventas_por_categoria = (
    ventas_clean
    .groupby("categoria", as_index=False)
    .agg(total=("total", "sum"), cantidad=("cantidad", "sum"),
         precio=("precio", "mean"), productos=("producto", "nunique"))
    .sort_values("total", ascending=False)
    .reset_index(drop=True)
)
ventas_por_categoria["participacion_%"] = ventas_por_categoria["total"] / total_general * 100
ventas_por_categoria["ingreso_x_producto"] = ventas_por_categoria["total"] / ventas_por_categoria["productos"]
ventas_por_categoria

In [ ]:
# Ingresos totales por categoría
fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(ventas_por_categoria["categoria"], ventas_por_categoria["total"], color="#4C78A8")
ax.bar_label(barras, labels=[f"${v:,.0f}" for v in ventas_por_categoria["total"]], padding=3)
ax.set_title("Ingresos totales por categoría")
ax.set_xlabel("Categoría")
ax.set_ylabel("Ingreso total ($)")
ax.set_ylim(0, ventas_por_categoria["total"].max() * 1.12)
plt.tight_layout()
plt.show()

### Unión con marketing

Marketing se agrupa por producto y se une a ventas. Como el nombre del producto se normalizó con la misma función en ambas tablas, la unión debería encontrar todos los productos. Lo verifico en vez de suponerlo.

In [ ]:
marketing_por_producto = (
    marketing_clean
    .groupby("producto", as_index=False)
    .agg(cantidad_campanias=("id_campanha", "count"), costo_total_marketing=("costo", "sum"))
)

ventas_marketing = pd.merge(
    ventas_por_producto, marketing_por_producto,
    on="producto", how="left", validate="one_to_one", indicator=True,
)

con_marketing = int((ventas_marketing["_merge"] == "both").sum())
sin_marketing = ventas_marketing.loc[ventas_marketing["_merge"] != "both", "producto"].tolist()
solo_marketing = sorted(set(marketing_por_producto["producto"]) - set(ventas_por_producto["producto"]))

print(f"Productos con datos de marketing: {con_marketing} de {len(ventas_marketing)}")
print("Sin marketing:", sin_marketing or "ninguno")
print("En marketing pero sin ventas:", solo_marketing or "ninguno")

ventas_marketing = ventas_marketing.drop(columns="_merge")
ventas_marketing.head(10)

## 5. Concentración de ingresos (Pareto)

El principio de Pareto dice que cerca del 80% de los resultados viene del 20% de las causas. Lo contrasto con los datos: ¿cuántos productos hacen falta para llegar al 80% de los ingresos?

In [ ]:
pareto = ventas_por_producto[["producto", "total"]].copy()
pareto["porcentaje"] = pareto["total"] / total_general * 100
pareto["porcentaje_acumulado"] = pareto["porcentaje"].cumsum()

n_productos = len(pareto)
n_para_80 = int((pareto["porcentaje_acumulado"] < 80).sum() + 1)
n_top20 = round(n_productos * 0.20)
share_top20 = pareto["porcentaje"].head(n_top20).sum()

print(f"El 20% de los productos ({n_top20} de {n_productos}) genera el {share_top20:.1f}% de los ingresos.")
print(f"Para llegar al 80% de los ingresos hacen falta {n_para_80} productos ({n_para_80 / n_productos:.0%}).")

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(pareto["producto"], pareto["porcentaje"], color="#4C78A8")
ax.set_ylabel("% de los ingresos")
ax.tick_params(axis="x", rotation=90)
ax2 = ax.twinx()
ax2.plot(pareto["producto"], pareto["porcentaje_acumulado"], color="#E45756", marker="o", label="% acumulado")
ax2.axhline(80, color="gray", linestyle="--", label="80%")
ax2.set_ylim(0, 125)
ax2.set_yticks(range(0, 101, 20))
ax2.set_ylabel("% acumulado")
ax2.grid(False)
ax.grid(axis="x", visible=False)
ax2.legend(loc="upper right", ncol=2)
ax.set_title("Pareto de ingresos por producto")
plt.tight_layout()
plt.show()

In [ ]:
# Productos de alto rendimiento: ingreso total en el percentil 80 o más
p80 = ventas_por_producto["total"].quantile(0.80)
ventas_top = ventas_por_producto[ventas_por_producto["total"] >= p80]
print(f"Umbral (percentil 80): ${p80:,.2f} -> {len(ventas_top)} productos")
ventas_top[["producto", "categoria", "total", "cantidad", "precio"]]

## 6. Estadística descriptiva y outliers

Todas las medidas se calculan sobre el **ingreso total por producto**. Al ser todos valores distintos, la moda no aporta información y no se usa.

In [ ]:
serie = ventas_por_producto["total"]

media, mediana = serie.mean(), serie.median()
desvio, varianza = serie.std(ddof=1), serie.var(ddof=1)
minimo, maximo = serie.min(), serie.max()
rango = maximo - minimo
cv = desvio / media
asimetria = serie.skew()

resumen = pd.Series({
    "Media": media, "Mediana": mediana, "Desvío estándar": desvio,
    "Coef. de variación": cv, "Mínimo": minimo, "Máximo": maximo,
    "Rango": rango, "Asimetría": asimetria,
})
resumen.to_frame("Ingreso total por producto")

In [ ]:
# Rango intercuartílico (IQR) y outliers con la regla 1.5 × IQR
q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers_altos = ventas_por_producto[serie > lim_sup]
outliers_bajos = ventas_por_producto[serie < lim_inf]

print(f"Q1: {q1:,.2f} | Q3: {q3:,.2f} | IQR: {iqr:,.2f}")
print(f"Límites: [{lim_inf:,.2f} ; {lim_sup:,.2f}]")
print(f"\nOutliers altos ({len(outliers_altos)}): {', '.join(outliers_altos['producto'])}")
print(f"Outliers bajos ({len(outliers_bajos)}): {', '.join(outliers_bajos['producto'])}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.boxplot(x=serie, ax=axes[0], color="#9ecae9")
axes[0].set_title("Dispersión del ingreso total por producto")
axes[0].set_xlabel("Ingreso total ($)")

sns.histplot(serie, kde=True, bins=10, ax=axes[1], color="#4C78A8")
axes[1].axvline(media, color="#E45756", linestyle="--", label=f"Media: ${media:,.0f}")
axes[1].axvline(mediana, color="black", linestyle=":", label=f"Mediana: ${mediana:,.0f}")
axes[1].set_title("Distribución del ingreso total por producto")
axes[1].set_xlabel("Ingreso total ($)")
axes[1].set_ylabel("Cantidad de productos")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Ingresos por producto, coloreados por categoría
orden = ventas_por_producto.sort_values("total", ascending=False)
plt.figure(figsize=(12, 6))
sns.barplot(data=orden, x="producto", y="total", hue="categoria", dodge=False)
plt.axhline(media, color="gray", linestyle="--", label="Media")
plt.xticks(rotation=90)
plt.title("Ingresos totales por producto")
plt.xlabel("Producto")
plt.ylabel("Ingreso total ($)")
plt.legend(title="Categoría")
plt.tight_layout()
plt.show()

In [ ]:
# Ingreso por venta individual, según categoría
plt.figure(figsize=(10, 5))
sns.boxplot(data=ventas_clean, x="categoria", y="total", hue="categoria", legend=False)
plt.title("Ingreso por venta, por categoría")
plt.xlabel("Categoría")
plt.ylabel("Ingreso por venta ($)")
plt.tight_layout()
plt.show()

## 7. Correlaciones

¿Los productos más caros se venden más o menos? Calculo la correlación de Pearson (relación lineal) y la de Spearman (por rangos, más robusta), junto con su p-valor. Con solo 30 productos, el p-valor es clave para no sobreinterpretar.

In [ ]:
def describir_correlacion(r, p):
    fuerza = ("muy débil o nula" if abs(r) < 0.2 else "débil" if abs(r) < 0.4
              else "moderada" if abs(r) < 0.7 else "fuerte")
    sentido = "positiva" if r > 0 else "negativa"
    signif = ("estadísticamente significativa" if p < 0.05
              else "NO estadísticamente significativa (puede deberse al azar)")
    return f"correlación {sentido} {fuerza}, {signif}"


r_pearson, p_pearson = stats.pearsonr(ventas_por_producto["precio"], ventas_por_producto["cantidad"])
r_spearman, p_spearman = stats.spearmanr(ventas_por_producto["precio"], ventas_por_producto["cantidad"])
precio_min, precio_max = ventas_por_producto["precio"].min(), ventas_por_producto["precio"].max()

print(f"Pearson : r = {r_pearson:.3f} (p = {p_pearson:.3f}) -> {describir_correlacion(r_pearson, p_pearson)}")
print(f"Spearman: r = {r_spearman:.3f} (p = {p_spearman:.3f})")
print(f"Rango de precios promedio entre productos: ${precio_min:,.2f} a ${precio_max:,.2f}")

plt.figure(figsize=(8, 5))
sns.scatterplot(data=ventas_por_producto, x="precio", y="cantidad", hue="categoria", s=80)
plt.title(f"Precio promedio vs. unidades vendidas (r = {r_pearson:.2f})")
plt.xlabel("Precio promedio ($)")
plt.ylabel("Unidades vendidas")
plt.tight_layout()
plt.show()

In [ ]:
# Mapa de calor de correlaciones
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(ventas_por_producto[["precio", "cantidad", "total"]].corr(),
            annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlaciones entre variables (por producto)")
plt.tight_layout()
plt.show()

## 8. Evolución de ingresos en el tiempo

Arriba, el ingreso diario con su media móvil de 7 días (para ver la tendencia sin el ruido). Abajo, el ingreso acumulado. Van en gráficos separados porque tienen escalas muy distintas.

In [ ]:
ingresos_por_dia = ventas_clean.groupby("fecha_venta")["total"].sum().sort_index()
ingresos_acumulados = ingresos_por_dia.cumsum()
media_movil = ingresos_por_dia.rolling(7).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
ax1.plot(ingresos_por_dia, color="skyblue", label="Ingreso diario")
ax1.plot(media_movil, color="#E45756", linewidth=2, label="Media móvil 7 días")
ax1.set_ylabel("Ingreso diario ($)")
ax1.legend()
ax1.set_title("Ingreso diario")

ax2.plot(ingresos_acumulados, color="royalblue", linewidth=2)
ax2.set_ylabel("Ingreso acumulado ($)")
ax2.set_xlabel("Fecha de venta")
ax2.set_title("Ingreso acumulado")
plt.tight_layout()
plt.show()

dias_con_ventas = len(ingresos_por_dia)
promedio_diario = ingresos_por_dia.mean()
mejor_dia, valor_mejor_dia = ingresos_por_dia.idxmax(), ingresos_por_dia.max()
peor_dia, valor_peor_dia = ingresos_por_dia.idxmin(), ingresos_por_dia.min()
ingresos_por_mes = ventas_clean.groupby(ventas_clean["fecha_venta"].dt.to_period("M"))["total"].sum()
mejor_mes, peor_mes = ingresos_por_mes.idxmax(), ingresos_por_mes.idxmin()
mayor_pausa = int(ingresos_por_dia.index.to_series().diff().dt.days.max())

print(f"Días con ventas: {dias_con_ventas} | Promedio diario: ${promedio_diario:,.2f}")
print(f"Mejor día: {mejor_dia:%d/%m/%Y} (${valor_mejor_dia:,.2f}) | Peor día: {peor_dia:%d/%m/%Y} (${valor_peor_dia:,.2f})")
print(f"Mejor mes: {mejor_mes.strftime("%m/%Y")} (${ingresos_por_mes.max():,.0f}) | Peor mes: {peor_mes.strftime("%m/%Y")} (${ingresos_por_mes.min():,.0f})")
print(f"Mayor pausa entre días con ventas: {mayor_pausa} día(s)")

## 9. Marketing vs. ingresos

¿Los productos con más inversión en marketing venden más? Es un análisis exploratorio: son solo 30 productos, y cada uno tiene prácticamente la misma cantidad de campañas.

In [ ]:
mk = ventas_marketing.dropna(subset=["costo_total_marketing"])
r_mk, p_mk = stats.pearsonr(mk["costo_total_marketing"], mk["total"])
costo_min, costo_max = mk["costo_total_marketing"].min(), mk["costo_total_marketing"].max()

print(f"Pearson costo de marketing vs. ingreso total: r = {r_mk:.3f} (p = {p_mk:.3f}) -> {describir_correlacion(r_mk, p_mk)}")
print(f"Costo total de marketing por producto: de {costo_min:,.2f} a {costo_max:,.2f}")
print(f"Ingreso total por producto: de {mk['total'].min():,.2f} a {mk['total'].max():,.2f}")

plt.figure(figsize=(8, 5))
sns.scatterplot(data=mk, x="costo_total_marketing", y="total", hue="categoria", s=80)
plt.title(f"Costo de marketing vs. ingreso total por producto (r = {r_mk:.2f})")
plt.xlabel("Costo total de marketing")
plt.ylabel("Ingreso total ($)")
plt.tight_layout()
plt.show()

> ⚠️ **Ojo con la escala:** los costos de marketing son del orden de decenas y los ingresos del orden de decenas de miles. Sin un diccionario de datos no sé si el costo está en otra unidad (por ejemplo, miles) o si es realmente tan bajo, por eso **no calculo un ROI**: el resultado sería engañoso.

## 10. Dashboard interactivo (Plotly)

Un solo gráfico con un selector para alternar entre ingresos, unidades y precio promedio de los 10 productos con más ingresos. El archivo HTML queda guardado para poder compartirlo (los gráficos de Plotly no se visualizan dentro de GitHub).

In [ ]:
import plotly.graph_objects as go

top10 = ventas_por_producto.head(10)
metricas = [("total", "Ingresos totales ($)"),
            ("cantidad", "Unidades vendidas"),
            ("precio", "Precio promedio ($)")]

fig = go.Figure()
for i, (col, nombre) in enumerate(metricas):
    fig.add_bar(
        x=top10["producto"], y=top10[col], name=nombre, visible=(i == 0),
        marker_color="#4C78A8",
        hovertemplate="%{x}<br>" + nombre + ": %{y:,.2f}<extra></extra>",
    )

botones = [
    dict(label=nombre, method="update",
         args=[{"visible": [j == i for j in range(len(metricas))]},
               {"yaxis.title.text": nombre}])
    for i, (col, nombre) in enumerate(metricas)
]

fig.update_layout(
    title="Dashboard de ventas: top 10 productos por ingresos",
    xaxis_title="Producto", yaxis_title=metricas[0][1],
    xaxis_tickangle=-45, height=550, showlegend=False,
    updatemenus=[dict(buttons=botones, direction="down", showactive=True, x=0, y=1.15)],
)

fig.write_html("dashboard_ventas.html", include_plotlyjs="cdn")
fig.show()

## 11. Conclusiones

Los números de esta sección se calculan a partir de las variables del análisis, por lo que siempre coinciden con lo que muestran las tablas y los gráficos de arriba.

In [ ]:
top3 = ventas_por_producto.head(3)
bottom3 = ventas_por_producto.tail(3)
lider, ultima = ventas_por_categoria.iloc[0], ventas_por_categoria.iloc[-1]
dif_categorias = (lider["total"] / ultima["total"] - 1) * 100
dif_media_mediana = (media / mediana - 1) * 100
forma = ("aproximadamente simétrica" if abs(asimetria) < 0.5
         else "sesgada hacia valores altos" if asimetria > 0 else "sesgada hacia valores bajos")
nombres = lambda df: ", ".join(df["producto"])

ranking_cat = " · ".join(
    f"**{r['categoria']}** ${r['total']:,.0f} ({r['participacion_%']:.1f}%)"
    for _, r in ventas_por_categoria.iterrows()
)

if share_top20 >= 70:
    texto_pareto = "La concentración es alta: se aproxima a la regla 80/20."
else:
    texto_pareto = ("**No se cumple la regla 80/20**: los ingresos están repartidos de forma bastante pareja "
                    "entre muchos productos, y ninguno es indispensable por sí solo.")

display(Markdown(f'''
### Resumen de los datos
- **{len(ventas_clean):,} ventas** limpias de **{n_productos} productos** en **{ventas_por_categoria.shape[0]} categorías**, entre el {ventas_clean["fecha_venta"].min():%d/%m/%Y} y el {ventas_clean["fecha_venta"].max():%d/%m/%Y}.
- **Ingresos totales: ${total_general:,.2f}**.
- Limpieza: {n_duplicados} duplicados eliminados, {n_precio_nulo} precios imputados y {n_cantidad_nula} filas descartadas por no tener cantidad.
- Para unir marketing con ventas hubo que normalizar los nombres de producto con la misma regla en ambas tablas; así se unieron **{con_marketing} de {len(ventas_marketing)} productos**.

### Distribución de los ingresos
- Ingreso medio por producto: **${media:,.0f}**; mediana: **${mediana:,.0f}** (la media difiere un {abs(dif_media_mediana):.1f}% de la mediana). La distribución es {forma} (asimetría = {asimetria:.2f}).
- Desvío estándar: **${desvio:,.0f}** (coeficiente de variación: {cv:.0%}). Rango: de ${minimo:,.0f} a ${maximo:,.0f}.
- Outliers según la regla del IQR: **{len(outliers_altos)} altos** ({nombres(outliers_altos) or "ninguno"}) y **{len(outliers_bajos)} bajos** ({nombres(outliers_bajos) or "ninguno"}). Los bajos importan tanto como los altos: son productos con ventas muy por debajo de lo normal.
- Pareto: el 20% de los productos ({n_top20}) aporta el **{share_top20:.1f}%** de los ingresos y hacen falta **{n_para_80} productos ({n_para_80 / n_productos:.0%})** para llegar al 80%. {texto_pareto}

### Categorías
- {ranking_cat}.
- La categoría líder supera a la última en un **{dif_categorias:.1f}%**: {"una diferencia chica; las categorías rinden de forma muy parecida." if dif_categorias < 10 else "una diferencia relevante."}

### Precio y unidades
- Correlación precio–unidades: r = **{r_pearson:.2f}** (p = {p_pearson:.2f}), {describir_correlacion(r_pearson, p_pearson)}.
- Los precios promedio de los productos son muy parecidos entre sí (de ${precio_min:,.2f} a ${precio_max:,.2f}), así que estos datos **no permiten afirmar que el precio ayude o perjudique las ventas**.

### Marketing
- Correlación costo de marketing–ingresos: r = **{r_mk:.2f}** (p = {p_mk:.2f}), {describir_correlacion(r_mk, p_mk)}.
- Los costos (entre {costo_min:,.2f} y {costo_max:,.2f} por producto) están en una escala muy distinta a la de los ingresos; hay que confirmar su unidad antes de calcular un retorno de la inversión.

### Evolución en el tiempo
- Promedio diario de **${promedio_diario:,.0f}** en {dias_con_ventas} días con ventas; el mejor día fue el {mejor_dia:%d/%m/%Y} (${valor_mejor_dia:,.0f}) y el peor el {peor_dia:%d/%m/%Y} (${valor_peor_dia:,.0f}).
- Mejor mes: **{mejor_mes.strftime("%m/%Y")}**; peor mes: **{peor_mes.strftime("%m/%Y")}**. La mayor pausa entre días con ventas fue de {mayor_pausa} día(s).

### Qué haría con esto
1. **Mirar por producto y no por categoría:** las categorías rinden parecido, pero hay productos que se destacan ({nombres(top3)}) y otros que quedan muy atrás ({nombres(bottom3)}).
2. **Revisar los productos de menor ingreso** (precio, exposición, stock, campañas) antes de decidir si se mantienen o se reemplazan.
3. **No basar decisiones de precio en este análisis:** con precios tan parecidos, no hay variación suficiente para medir su efecto.
4. **Conseguir el diccionario de datos de marketing** y cruzar las fechas de cada campaña con las ventas diarias, que es la comparación que realmente mide el impacto.

### Limitaciones
- Solo 30 productos: las correlaciones tienen poca potencia estadística.
- `ventas` no incluye `id_cliente`, por lo que no se pudo analizar el comportamiento de los clientes.
- Se trabajó con ingresos, no con rentabilidad: no hay datos de costos de los productos.
'''))